# CineMetrix

# Phase 2 : Data Cleaning

## Goal

Prepare a clean and analysis-ready dataset while preserving data integrity.

In [97]:
import pandas as pd
import numpy as np

In [98]:
df = pd.read_csv("../data/raw/Netflix.csv")

In [99]:
df.shape

(7787, 12)

## Observation

This notebook will never modify the original dataset directly.

In [100]:
df_clean = df.copy()

## Why create a copy?

- Preserve raw data
- Ensure reproducibility
- Avoid accidental modifications

In [101]:
df_clean.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration        int64
genres            str
description       str
dtype: object

## Observation

The `date_added` column is stored as string.

For temporal analysis, it should be converted into datetime.

In [102]:
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], errors='coerce')

C:\Users\HP\AppData\Local\Temp\ipykernel_12152\3582359811.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], errors='coerce')


In [103]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       7787 non-null   str           
 1   type          7787 non-null   str           
 2   title         7787 non-null   str           
 3   director      5398 non-null   str           
 4   cast          7069 non-null   str           
 5   country       7280 non-null   str           
 6   date_added    7777 non-null   datetime64[us]
 7   release_year  7787 non-null   int64         
 8   rating        7780 non-null   str           
 9   duration      7787 non-null   int64         
 10  genres        7787 non-null   str           
 11  description   7787 non-null   str           
dtypes: datetime64[us](1), int64(2), str(9)
memory usage: 730.2 KB


## Observation

The conversion was successful.

`date_added` is now datetime and can be used for time-based analysis.

In [104]:
missing = (
    df_clean.isnull().sum().to_frame("Missing Count")
)

missing["Missing %"] = ( 
    missing["Missing Count"]/len(df_clean)
)*100

missing.sort_values(
    "Missing %",
    ascending=False
)

,Missing Count,Missing %
director,2389,30.679337
cast,718,9.220496
country,507,6.510851
date_added,10,0.128419
rating,7,0.089893
title,0,0.000000
show_id,0,0.000000
type,0,0.000000
release_year,0,0.000000
duration,0,0.000000


In [105]:
df_clean.describe(include="all")

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description
count,7787,7787,7787,5398,7069,7280,7777,7787.000000,7780,7787.000000,7787,7787
unique,7787,2,7787,4049,6831,681,NaN,NaN,14,NaN,492,7769
top,s1,Movie,3%,"Raúl Campos, Jan Suter",David Attenborough,United States,NaN,NaN,TV-MA,NaN,Documentaries,Multiple women report their husbands as missin...
freq,1,5377,1,18,18,2555,NaN,NaN,2863,NaN,334,3
mean,NaN,NaN,NaN,NaN,NaN,NaN,2019-01-02 19:20:57.708628,2013.932580,NaN,69.122769,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,2008-01-01 00:00:00,1925.000000,NaN,1.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-01 00:00:00,2013.000000,NaN,2.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2019-03-08 00:00:00,2017.000000,NaN,88.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-20 00:00:00,2018.000000,NaN,106.000000,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-16 00:00:00,2021.000000,NaN,312.000000,NaN,NaN


In [106]:
pd.crosstab(
    df_clean["type"],
    df_clean["director"].isnull()
)

director,False,True
type,,
Movie,5214,163
TV Show,184,2226


## Observation

Director information is largely missing for TV Shows but mostly available for Movies.

Therefore, the missingness is systematic rather than random.

Decision:
The column will be preserved without imputation.

In [107]:
pd.crosstab(
    df_clean['type'],
    df_clean['cast'].isnull()
)

cast,False,True
type,,
Movie,4951,426
TV Show,2118,292


In [108]:
pd.crosstab(
    df_clean['type'],
    df_clean['country'].isnull()
)

country,False,True
type,,
Movie,5147,230
TV Show,2133,277


In [109]:
print('Cast :Movie missing ', 426/5377 *100,'TV show missing ', 292/2410 *100)

Cast :Movie missing  7.922633438720476 TV show missing  12.116182572614107


In [110]:
print('country:Movie missing ', 230/5377*100,'TV show missing ', 277/2410*100)

country:Movie missing  4.277478147665985 TV show missing  11.493775933609959


#### Cast : Movie missing = 7.9226% and Tv show missing = 12.1162% ,
#### country: Movie missing = 4.2775% and Tv show missing = 11.4937%

In [111]:
df_clean['country'].str.contains(",",na = False).sum()

np.int64(1153)

In [112]:
df_clean["genres"].str.contains(",", na=False).sum()

np.int64(5986)

## Observation

The genres column is a multi-label categorical variable.

It should not be treated as a single category during analysis.

In [113]:
df_clean["movie_minutes"] = np.where(
    df_clean["type"]=="Movie",
    df_clean["duration"],
    np.nan
)

df_clean["seasons"] = np.where(
    df_clean["type"]=="TV Show",
    df_clean["duration"],
    np.nan
)

In [114]:
df_clean.groupby("type")["duration"].describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
Movie,5377.0,99.307978,28.530881,3.0,86.0,98.0,114.0,312.0
TV Show,2410.0,1.775934,1.596359,1.0,1.0,1.0,2.0,16.0


In [115]:
df_clean[["type", "duration", "movie_minutes", "seasons"]].head(10)

,type,duration,movie_minutes,seasons
0,TV Show,4,NaN,4.0
1,Movie,143,143.0,NaN
2,Movie,124,124.0,NaN
3,Movie,90,90.0,NaN
4,TV Show,1,NaN,1.0
5,Movie,90,90.0,NaN
6,Movie,94,94.0,NaN
7,Movie,112,112.0,NaN
8,Movie,129,129.0,NaN
9,Movie,85,85.0,NaN


In [116]:
print(df_clean["movie_minutes"].describe())
print(df_clean["seasons"].describe())

count    5377.000000
mean       99.307978
std        28.530881
min         3.000000
25%        86.000000
50%        98.000000
75%       114.000000
max       312.000000
Name: movie_minutes, dtype: float64
count    2410.000000
mean        1.775934
std         1.596359
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        16.000000
Name: seasons, dtype: float64


## Observation

The original duration column represented two different concepts:

Movie duration (minutes),
TV Show duration (seasons)

This violates the tidy data principle that a column should represent a single variable.

Action Taken

Two new variables were created:

movie_minutes and 
seasons

allowing movie runtime analysis and TV show season analysis independently.

In [117]:
genre_split = df_clean['genres'].str.split(', ')
genre_exploded = (
    df_clean
    .assign(genres=genre_split)
    .explode('genres')
    .reset_index(drop=True)
)
genre_exploded.to_csv('../data/processed/netflix_genre_exploded.csv', index=False)

## Important Note

genre_exploded should only be used for genre-level analysis.

Since exploding duplicates rows for multi-genres, overall KPIs such as:

Total genres

must always be computed from df_clean, not from genre_exploded.

In [118]:
df['genres'].value_counts().head(10)

genres
Documentaries                                       334
Stand-Up Comedy                                     321
Dramas, International Movies                        320
Comedies, Dramas, International Movies              243
Dramas, Independent Movies, International Movies    215
Kids' TV                                            205
Children & Family Movies                            177
Documentaries, International Movies                 172
Children & Family Movies, Comedies                  169
Comedies, International Movies                      161
Name: count, dtype: int64

In [119]:
genre_exploded['genres'].value_counts().head(10)

genres
International Movies        2437
Dramas                      2106
Comedies                    1471
International TV Shows      1199
Documentaries                786
Action & Adventure           721
TV Dramas                    704
Independent Movies           673
Children & Family Movies     532
Romantic Movies              531
Name: count, dtype: int64

In [120]:
# split the composite string into a list per row
country_split = df_clean['country'].str.split(', ')

# explode turns each list element into its own row,
# duplicating every other column's value across the new rows
country_exploded = (
    df_clean
    .assign(country=country_split)
    .explode('country')
    .reset_index(drop=True)
)

In [121]:
print("Original unique country-strings (the buggy count):", df_clean['country'].nunique())
print("True unique countries after explode (the correct count):", country_exploded['country'].nunique())

old_top5 = df_clean['country'].value_counts().head(5)
new_top5 = country_exploded['country'].value_counts().head(5)
print(old_top5, "\n\n", new_top5)

Original unique country-strings (the buggy count): 681
True unique countries after explode (the correct count): 121
country
United States     2555
India              923
United Kingdom     397
Japan              226
South Korea        183
Name: count, dtype: int64 

 country
United States     3296
India              990
United Kingdom     722
Canada             412
France             349
Name: count, dtype: int64


### Why Country Explosion Was Necessary

The original `country` column contained composite values such as
"United States, United Kingdom".

Without normalization, these combinations are treated as separate categories,
leading to severe underestimation of individual countries.

After exploding the country column, the number of unique countries decreased
from 681 composite combinations to 121 true countries.

In [122]:
country_exploded.to_csv('../data/processed/netflix_country_exploded.csv', index=False)

## Important Note

country_exploded should only be used for country-level analysis.

Since exploding duplicates rows for multi-country titles, overall KPIs such as:

Total Titles
Movies
TV Shows

must always be computed from df_clean, not from country_exploded.

# Cleaning Decisions

## Decision 1

Convert date_added to datetime

Reason:
Required for temporal analysis.

----------------------------

## Decision 2

Do not remove missing directors

Reason:
30% missing.
Removing them would discard valuable observations.

----------------------------

## Decision 3

Cast and Country missing value , like director mostly for tv shows , 
(cast : 7.9% missing for Movies vs 12.1% for Tv shows ;Country : 4.3% vs 11.5%).
The skew is smaller than director's , we can treat it as systematic rather than random.
 No imputation required. Rows with missing cast/country are excluded only for analyses that specifically require that field.

----------------------------

## Decision 4

Country and genres contain composite values.
Also they are exploded and saved.



In [123]:
# Create processed dataset

df_clean.to_csv(
    "../data/processed/netflix_clean.csv",
    index=False
)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!
